# Semantic map

Map the semantic structure of the full publication corpus and compare papers with and without explicit UK Biobank mentions.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import shared_paths as P
from utils.data_analysis_00_dataset_analysis import (
    CATEGORY_PATTERNS,
    MODEL_NAMES,
    contains_pattern,
    load_publications,
    model_agreement_columns,
    normalise_bool,
    normalized_rows,
    output_dirs,
    sample_balanced,
    save_figure,
)

P.bootstrap()

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score


In [ ]:
df = load_publications(P.SHOWCASE_PLUS)
df.shape


In [ ]:
MAX_PER_GROUP = 3000
RANDOM_STATE = 42
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

sample = sample_balanced(df, "explicit_ukb_mention", MAX_PER_GROUP, RANDOM_STATE)
texts = sample["analysis_text"].str.slice(0, 3500).tolist()

try:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = model.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
    embedding_method = EMBEDDING_MODEL
except Exception:
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=12000)
    matrix = vectorizer.fit_transform(texts)
    components = min(100, matrix.shape[0] - 1, matrix.shape[1] - 1)
    embeddings = normalized_rows(TruncatedSVD(n_components=components, random_state=RANDOM_STATE).fit_transform(matrix))
    embedding_method = "TF-IDF + TruncatedSVD"

labels = sample["explicit_ukb_mention"].astype(int).to_numpy()
coordinates = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(embeddings)
sample["semantic_x"] = coordinates[:, 0]
sample["semantic_y"] = coordinates[:, 1]
silhouette = silhouette_score(embeddings, labels, metric="cosine") if len(np.unique(labels)) > 1 else np.nan


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_analysis_04_semantic_map")
sample.to_csv(table_dir / "semantic_sample_with_coordinates.csv", index=False)
pd.DataFrame(
    [{"embedding_method": embedding_method, "sample_size": len(sample), "silhouette_index_cosine": silhouette}]
).to_csv(table_dir / "semantic_metrics.csv", index=False)

figure, axis = plt.subplots(figsize=(9, 7))
for explicit, group in sample.groupby("explicit_ukb_mention"):
    label = "Explicit UKB mention" if explicit else "No explicit UKB mention"
    axis.scatter(group["semantic_x"], group["semantic_y"], s=12, alpha=0.55, label=f"{label} (n={len(group):,})")
axis.set(xlabel="Semantic component 1", ylabel="Semantic component 2", title=f"Semantic map (silhouette={silhouette:.4f})")
axis.grid(alpha=0.25)
axis.legend()
save_figure(figure, figure_dir / "semantic_map_explicit_ukb_mentions.png")
sample.head()
